<a href="https://colab.research.google.com/github/affreen/affreen_ai_ml/blob/create_model/05_practice_neural_network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [107]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [131]:
# Create a model class that inherits nn.module

class Model(nn.Module):
    def __init__(self, in_features=3, h1=5, h2=2, out_features=2):
        super().__init__()
        self.fc1 = nn.Linear(in_features, h1)
        self.fc2 = nn.Linear(h1, h2)
        self.out = nn.Linear(h2, out_features)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.out(x)
        return x


In [132]:

# Pick a manual seed for randomization
torch.manual_seed(41)

# Create an instance of the model
model = Model()


In [169]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/csv/all_compound_list.csv'
df = pd.read_csv(file_path)
df

df['Bonds_Type'] = df['Bonds_Type'].replace('Covalent', 0.0)
df['Bonds_Type'] = df['Bonds_Type'].replace('Ionic', 1.0)
df



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipython-input-450486453.py:14: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['Bonds_Type'] = df['Bonds_Type'].replace('Ionic', 1.0)


,Contains_Metal,Molecular_Weight,Bonds_Type,Organic
0,0,16.0,0.0,1
1,0,18.0,0.0,0
2,1,58.0,1.0,0
3,0,44.0,0.0,1
4,0,36.5,0.0,0
5,0,180.0,0.0,1
6,0,17.0,0.0,0
7,1,100.0,1.0,0
8,0,78.0,0.0,1
9,0,46.0,0.0,1


In [134]:
# Train test and split ! Set X and y

X = df.drop('Organic', axis=1)
X
y = df['Organic']
y

# Convert teh data frames into numpy arrays

X = X.values
y = y.values

In [162]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=33)

X_train = torch.FloatTensor(X_train)
X_test = torch.FloatTensor(X_test)
y_train = torch.LongTensor(y_train)
y_test = torch.LongTensor(y_test)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [163]:

epochs = 500
losses = []

for i in range(epochs):
    i += 1

    y_pred = model.forward(X_train) # Get predicted data

    loss = criterion(y_pred, y_train) # Measure the error
    losses.append(loss)

    # Keep track of the losses
    losses.append(loss.detach().numpy())

    # print every 10 epochs
    if i % 10 == 0:
      print(f'Epoch: {i} Loss: {loss}')

    # do some back propagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()




Epoch: 10 Loss: 0.094200998544693
Epoch: 20 Loss: 0.09403575211763382
Epoch: 30 Loss: 0.09404191374778748
Epoch: 40 Loss: 0.09404083341360092
Epoch: 50 Loss: 0.09403426200151443
Epoch: 60 Loss: 0.09403127431869507
Epoch: 70 Loss: 0.09402965754270554
Epoch: 80 Loss: 0.09402838349342346
Epoch: 90 Loss: 0.0940273106098175
Epoch: 100 Loss: 0.09402628988027573
Epoch: 110 Loss: 0.09402531385421753
Epoch: 120 Loss: 0.09402421861886978
Epoch: 130 Loss: 0.09402317553758621
Epoch: 140 Loss: 0.09402204304933548
Epoch: 150 Loss: 0.09402094036340714
Epoch: 160 Loss: 0.09401974827051163
Epoch: 170 Loss: 0.09401857852935791
Epoch: 180 Loss: 0.0940173789858818
Epoch: 190 Loss: 0.09401613473892212
Epoch: 200 Loss: 0.09401487559080124
Epoch: 210 Loss: 0.09401359409093857
Epoch: 220 Loss: 0.0940123125910759
Epoch: 230 Loss: 0.09401098638772964
Epoch: 240 Loss: 0.09400966018438339
Epoch: 250 Loss: 0.09400831907987595
Epoch: 260 Loss: 0.0940069928765297
Epoch: 270 Loss: 0.09400562196969986
Epoch: 280 Loss:

In [164]:
# Evaluate

with torch.no_grad(): # turn off back propagation
    y_eval = model.forward(X_test)
    loss = criterion(y_eval, y_test)
loss

tensor(0.6968)

In [165]:
correct = 0

with torch.no_grad():
    for i, data in enumerate(X_test):
        y_val = model.forward(data)

        print(f'{i+1}. {str(y_val)} \t {y_test[i]}')

        # correct or not
        if y_val.argmax().item() == y_test[i]:
            correct += 1

print(f'We got so many correct: {correct}')


1. tensor([ 3.9153, -3.7674]) 	 0
2. tensor([ 3.6874, -3.5317]) 	 0
3. tensor([-4.7228,  5.1655]) 	 1
4. tensor([-19.0930,  20.0264]) 	 1
5. tensor([-1.9086,  2.2553]) 	 0
6. tensor([-6.8783,  7.3947]) 	 1
We got so many correct: 5


In [177]:
# Feed new data to make predictions

new_compund = torch.tensor([0, 17.0, 0.0])

with torch.no_grad():
    print(model(new_compund))

tensor([ 0.4266, -0.1596])
